# Agricultural Insights from Tirunelveli District

Generated starter notebook from [DesiData](https://www.desidata.in) — India's open dataset catalogue. Run the cells top to bottom.

The agricultural data from Tirunelveli district provides a detailed look at crop production across three distinct seasons. During the Kar season (April-July), the area cultivated was 4075 hectares, yielding 19074 metric tons of production at a rate of 4680 kilograms per hectare. In the Pishanam season (August-November), the cultivated area expanded to 12124 hectares, producing 55265 metric tons with a yield of 4558 kilograms per hectare. The Navarai season (December-March) saw the largest cultivated area of 23577 hectares, resulting in 102623 metric tons of production and a yield of 4353 kilograms per hectare. Overall, the district's total agricultural area was 39776 hectares, with a production of 176962 metric tons and an average yield of 4449 kilograms per hectare.

Source-supported facts: The district is Tirunelveli. During the Kar season (April-July), the area was 4075 ha, production was 19074 MTs, and yield was 4680 Kg/ha. During the Pishanam season (Aug-Nov), the area was 12124 ha, production was 55265 MTs, and yield was 4558 Kg/ha. During the Navarai season (Dec-March), the area was 23577 ha, production was 102623 MTs, and yield was 4353 Kg/ha. The district total area was 39776 ha, production was 176962 MTs, and yield was 4449 Kg/ha.

Source: agritech.tnau.ac.in  
Licence: Not specified  
Dataset page: https://www.desidata.in/datasets/agricultural-insights-from-tirunelveli-district

In [ ]:
!pip install pandas matplotlib --quiet

## Load the dataset

Dataset downloads require a free DD token. Create one in Profile & settings, then set `DD_TOKEN` in your environment.
In Colab, add `DD_TOKEN` under Secrets and enable notebook access to that secret.

In [ ]:
import io
import json
import os
import pandas as pd
import urllib.request

url = "https://www.desidata.in/api/datasets/agricultural-insights-from-tirunelveli-district/download"
if "COLAB_RELEASE_TAG" in os.environ and "DD_TOKEN" not in os.environ:
    from google.colab import userdata
    os.environ["DD_TOKEN"] = userdata.get("DD_TOKEN")
request = urllib.request.Request(url, headers={"Authorization": f"Bearer {os.environ['DD_TOKEN']}", "Accept": "application/json"})
with urllib.request.urlopen(request) as response:
    file_url = json.load(response)["download_url"]
# Fetch the signed file URL separately; never send DD_TOKEN to the storage host.
with urllib.request.urlopen(file_url) as response:
    df = pd.read_csv(io.BytesIO(response.read()))

print(f"{df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

## Explore

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

## See one example

Every row pairs a question with the text it was written from.

In [ ]:
row = df.iloc[0]

print("QUESTION:", row["question"])

print("ANSWER:", str(row["answer"])[:400])

print()
print("SOURCE CONTEXT:")
print(str(row["context"])[:600])

## Mini baseline — can TF-IDF find the source paragraph?

Each question was written from its row's source paragraph. A simple TF-IDF
retriever has to rank the correct one first. Its hit rate is the number
any trained model on this dataset must beat.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
import numpy as np

sample = df.head(300)
questions = sample["question"].astype(str).to_numpy()
passages = sample["context"].astype(str).to_numpy()

vectorizer = TfidfVectorizer(stop_words="english")
passage_vectors = vectorizer.fit_transform(passages)
question_vectors = vectorizer.transform(questions)

best = np.empty(len(questions), dtype=int)
for i in range(len(questions)):
    best[i] = linear_kernel(question_vectors[i], passage_vectors).argmax()

hit_rate = (passages[best] == passages).mean()
print(f"TF-IDF retrieval lands on the correct paragraph for {hit_rate:.0%} of questions")
print("That is the number a trained model has to beat.")

## What will you build?

- Compare yields or prices across crops, seasons or districts
- Model an outcome column against rainfall or year if they exist
- Combine this with weather or satellite data for a stronger model

- Browse more [Agriculture datasets](https://www.desidata.in/datasets?category=Agriculture) or the [full catalogue](https://www.desidata.in/datasets).
- Sharing your work? Tag DesiData — we feature community notebooks built on this catalogue.

In [ ]:
# Data via DesiData — India's open dataset catalogue
print("Dataset page:", "https://www.desidata.in/datasets/agricultural-insights-from-tirunelveli-district")